In [1]:
import json
import os
import numpy as np
import copy

# --- 1. 出力ディレクトリの準備 ---
# Jupyterのローカル環境（カレントディレクトリ）に "water_modtran_files" という名前のフォルダを作成します。
output_dir = r"E:\メタン\2025_HISUI_1_地獄の門\MOD\WATER"
os.makedirs(output_dir, exist_ok=True)
print(f"出力先フォルダ '{output_dir}' を準備しました。")

# --- 2. アップロードされた 'sample.json' の読み込み ---
# このコードセルと同じ場所にある "sample.json" を読み込みます
original_file_path = r"E:\メタン\2025_HISUI_1_地獄の門\MOD\sample.json"
h2o_values = np.arange(0.00, 5.01, 0.25) # 0.0, 0.25, ..., 5.00
try:
    with open(original_file_path, 'r', encoding='utf-8') as f:
        original_data = json.load(f)
    print(f"テンプレートファイル '{original_file_path}' を読み込みました。")

    # --- 3. ループ処理でファイルを作成 ---
    created_files_count = 0
    errors = False
    
    for value in h2o_values:
        # ファイル名とCSVPRNT用に値を文字列形式（小数点以下2桁）にする
        value_str = f"{value:.2f}"
        
        # 元のデータのディープコピーを作成して変更
        new_data = copy.deepcopy(original_data)
        
        # --- 4. JSONの内容を変更 (正しい構造のパス) ---
        
        # 構造 (data['MODTRAN'][0]['MODTRANINPUT']) にアクセス
        try:
            # MODTRAN[0].MODTRANINPUT の辞書部分を取得
            modtran_input_section = new_data['MODTRAN'][0]['MODTRANINPUT']
        except (KeyError, IndexError, TypeError) as e:
            print(f"エラー: 基本構造 'MODTRAN[0].MODTRANINPUT' が見つかりません。 {e}")
            errors = True
            break
            
        # a) H2OSTR の値を更新 (数値として)
        try:
            modtran_input_section['ATMOSPHERE']['H2OSTR'] = value
        except KeyError:
            print(f"エラー: 'ATMOSPHERE.H2OSTR' のパスが無効です。")
            errors = True
            break 
        
        # b) CSVPRNT の値を更新 (文字列として)
        # ユーザーの望む構造 (FILEOPTIONS の中に CSVPRNT) に合わせる
        try:
            # 'FILEOPTIONS' が存在するか確認し、なければ作成する
            if 'FILEOPTIONS' not in modtran_input_section:
                modtran_input_section['FILEOPTIONS'] = {} # 空の辞書を作成
                # print(f"'{value_str}.json' のために 'FILEOPTIONS' キーを作成しました。")
                
            modtran_input_section['FILEOPTIONS']['CSVPRNT'] = f"ch_h2ostr_{value_str}.csv"
            
            # テンプレート (0.25.json) に 'SPECTRAL.CSVPRNT' が存在していた場合、
            # それが 'FILEOPTIONS.CSVPRNT' と重複するため、
            # 望ましくない可能性のある 'SPECTRAL.CSVPRNT' を削除する
            if 'SPECTRAL' in modtran_input_section and 'CSVPRNT' in modtran_input_section['SPECTRAL']:
                del modtran_input_section['SPECTRAL']['CSVPRNT']
                # print(f"'{value_str}.json': 'SPECTRAL.CSVPRNT' を削除し、'FILEOPTIONS.CSVPRNT' に移動しました。")

        except KeyError:
             print(f"エラー: 'FILEOPTIONS' または 'SPECTRAL' の構造に問題があります。")
             errors = True
             break

        # --- 5. 新しいJSONファイルとして保存 ---
        new_filename = f"{value_str}.json"
        # 出力先フォルダ (water_modtran_files) 内に保存
        new_file_path = os.path.join(output_dir, new_filename)
        
        with open(new_file_path, 'w', encoding='utf-8') as f:
            json.dump(new_data, f, indent=4)
        
        created_files_count += 1
            
    if not errors:
        print(f"\n処理が完了しました。")
        print(f"{created_files_count} 個の .json ファイルが '{output_dir}' フォルダ内に作成されました。")
        print(f"（{h2o_values[0]:.2f}.json から {h2o_values[-1]:.2f}.json まで）")
    else:
        print("\nエラーが発生したため、処理を中断しました。")


except FileNotFoundError:
    print(f"エラー: 元となるファイル '{original_file_path}' が見つかりませんでした。")
    print("セッションに '0.25.json' がアップロードされているか確認してください。")
except json.JSONDecodeError:
    print(f"エラー: '{original_file_path}' は有効なJSONファイルではありません。")
except Exception as e:
    print(f"予期しないエラーが発生しました: {e}")

出力先フォルダ 'E:\メタン\2025_HISUI_1_地獄の門\MOD\WATER' を準備しました。
テンプレートファイル 'E:\メタン\2025_HISUI_1_地獄の門\MOD\sample.json' を読み込みました。

処理が完了しました。
21 個の .json ファイルが 'E:\メタン\2025_HISUI_1_地獄の門\MOD\WATER' フォルダ内に作成されました。
（0.00.json から 5.00.json まで）
